## Data Ingestion/Bronze Table

Data ingestion is the process of **bringing data from an external source into our data platform so it can be processed and analyzed later**.

For FlightPulse, the external source is the **OpenSky Network API**, which provides live aircraft state information such as aircraft identifiers, location, altitude, velocity, and flight status.

### What We Are Doing

The ingestion process:

1. Sends a request to the OpenSky API.
2. Receives the latest aircraft data as JSON.
3. Extracts the aircraft state records from the response.
4. Converts the records into a **Spark DataFrame**.
5. Applies an explicit schema to ensure consistent data types.
6. Stores the data as a **Delta table** in Databricks.

### Pipeline

```text
OpenSky Network
      ↓
API Request
      ↓
JSON Aircraft Data
      ↓
Spark DataFrame
      ↓
Delta Lake
      ↓
Bronze Layer
```

### Bronze Layer

The ingested data is stored in:

```text
workspace.default.bronze_aircraft
```

The Bronze layer represents the **initial/raw stage of the data pipeline**. At this stage, we primarily focus on capturing the data reliably rather than performing extensive transformations.

The data will later be cleaned and transformed in the **Silver layer**, followed by analytics-ready datasets in the **Gold layer**.

### Why We Are Doing This

Separating ingestion from transformation allows the original incoming data to be preserved and gives us a reliable starting point for the rest of the FlightPulse data pipeline.

### What gets cleaned

Nothing, on purpose: Bronze keeps OpenSky's data as it arrives, and
`02a_transform_silver_aircraft` does the cleaning. This notebook only:

| Step | What it does |
|---|---|
| **Explicit schema** | Gives every column a fixed type (e.g. `last_contact` as a number, `on_ground` as true/false), so Spark doesn't guess them differently from run to run |
| **Replace the table** | Overwrites `bronze_aircraft` with the latest snapshot |

### Known limitations

- **No history:** each hourly run replaces the previous snapshot, so past aircraft positions are not kept.
- **No snapshot time:** OpenSky's own snapshot time (`time` in the response) isn't saved, so how old the table is can only be told from `last_contact`.
- **An empty response fails the run:** if OpenSky returns no aircraft (`states` is null), creating the DataFrame raises an error.
- **Weather depends on aircraft:** the weather step below is in the same notebook, after the aircraft step, so a failed OpenSky call also means no weather that hour.


In [ ]:
import requests
from pyspark.sql.types import *


url = "https://opensky-network.org/api/states/all"

response = requests.get(url, timeout=30)
response.raise_for_status()

states = response.json()["states"]

columns = [
     "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source"
]

# Create a Spark DataFrame

schema = StructType([
    StructField("icao24",StringType(),True),
    StructField("callsign",StringType(),True),
    StructField("origin_country",StringType(),True),
    StructField("time_position",LongType(),True),
    StructField("last_contact",LongType(),True),
    StructField("longitude",DoubleType(),True),
    StructField("latitude",DoubleType(),True),
    StructField("baro_altitude",DoubleType(),True),
    StructField("on_ground",BooleanType(),True),
    StructField("velocity",DoubleType(),True),
    StructField("true_track",DoubleType(),True),
    StructField("vertical_rate",DoubleType(),True),
    StructField("sensors",ArrayType(IntegerType()),True),
    StructField("geo_altitude",DoubleType(),True),
    StructField("squawk",StringType(),True),
    StructField("spi",BooleanType(),True),
    StructField("position_source",IntegerType(),True)
])
spark_df = spark.createDataFrame(states, schema)


display(spark_df)

Now let's create the Bronze table and verify it.

In [ ]:
spark_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.bronze_aircraft")

In [ ]:
%sql
SELECT *
FROM workspace.default.bronze_aircraft
LIMIT 10;

### Weather history

Also saves the weather reports (METARs) for the tracked airports, Kuala Lumpur and
Penang, into `workspace.default.bronze_weather_history`, so delays can later be
compared with the weather at the time. The source is the Aviation Weather Center,
which is free and has no quota.

- Asks for the last 3 hours of reports each run, so a missed run leaves no gap.
- A report already saved (same airport and observation time) is not added again.
- If the weather service is down, this step is skipped so the aircraft data still updates.

#### What gets cleaned

| Step | Raw data problem | What this step does |
|---|---|---|
| **Numbers** | Temperature, dew point, wind and gust can be missing or not numeric | Converts them to decimals; anything unreadable becomes null |
| **Incomplete reports** | A report without an airport code or observation time can't be matched later | Drops it |
| **Times** | Observation time is in Unix seconds | Converts it to a timestamp |
| **Duplicates** | Asking for 3 hours each hour returns the same reports several times | Keeps one report per airport and observation time, within the run and against the history |

#### Not cleaned yet

- `visibility` stays as text, because it can be `"10+"`, and `wind_dir` can be `"VRB"` (variable). A number column for each would make them easier to analyse.
- **Corrected reports:** if an airport re-issues a report for the same time, the first version is kept, since the MERGE only inserts.


In [ ]:
import requests
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, timestamp_seconds, count, max
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# ICAO codes of the tracked airports: Kuala Lumpur, Penang.
WEATHER_AIRPORTS = ["WMKK", "WMKP"]
WEATHER_HISTORY = "workspace.default.bronze_weather_history"


def as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


try:
    response = requests.get(
        "https://aviationweather.gov/api/data/metar",
        params={"ids": ",".join(WEATHER_AIRPORTS), "format": "json", "hours": 3},
        timeout=30,
    )
    response.raise_for_status()
    metars = response.json() if response.status_code != 204 else []
except requests.RequestException as error:
    metars = []
    print("Weather skipped this run:", error)

weather_schema = StructType([
    StructField("icao", StringType(), False),
    StructField("observed_at", LongType(), False),
    StructField("temp_c", DoubleType(), True),
    StructField("dewpoint_c", DoubleType(), True),
    StructField("wind_dir", StringType(), True),        # degrees, or "VRB" when variable
    StructField("wind_kt", DoubleType(), True),
    StructField("gust_kt", DoubleType(), True),
    StructField("visibility", StringType(), True),      # miles, may be "10+"
    StructField("weather_codes", StringType(), True),   # e.g. "-RA BR" (light rain, mist)
    StructField("cloud_cover", StringType(), True),
    StructField("flight_category", StringType(), True), # VFR, MVFR, IFR or LIFR
    StructField("raw_report", StringType(), True),
])

weather_rows = [
    (
        m["icaoId"],
        int(m["obsTime"]),
        as_float(m.get("temp")),
        as_float(m.get("dewp")),
        None if m.get("wdir") is None else str(m.get("wdir")),
        as_float(m.get("wspd")),
        as_float(m.get("wgst")),
        None if m.get("visib") is None else str(m.get("visib")),
        m.get("wxString"),
        m.get("cover"),
        m.get("fltCat"),
        m.get("rawOb"),
    )
    for m in metars
    if m.get("icaoId") and m.get("obsTime") is not None
]

if weather_rows:
    reports = spark.createDataFrame(weather_rows, weather_schema) \
        .withColumn("observed_at", timestamp_seconds(col("observed_at"))) \
        .withColumn("ingested_at", current_timestamp()) \
        .dropDuplicates(["icao", "observed_at"])

    if not spark.catalog.tableExists(WEATHER_HISTORY):
        reports.write.format("delta").saveAsTable(WEATHER_HISTORY)
    else:
        DeltaTable.forName(spark, WEATHER_HISTORY).alias("history") \
            .merge(reports.alias("new"), "history.icao = new.icao AND history.observed_at = new.observed_at") \
            .whenNotMatchedInsertAll() \
            .execute()

    display(
        spark.table(WEATHER_HISTORY)
        .groupBy("icao")
        .agg(count("*").alias("reports"), max("observed_at").alias("latest"))
    )